# Recurrent Neural Net

In this notebook I will demonstrate how a Recurrent Neural Net (RNN) can be used to predict time series data very accurately.

### Import

As always we begin by importing all the required libraries.

In [ ]:
%matplotlib inline

from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import tensorflow as tf
import os

In [ ]:
csv_path = "./data/epidemic_process_raw_data.csv"
df = pd.read_csv(csv_path)
df.head(4)

In [ ]:

dfSusceptible = df[df.index % 4 == 0]
dfInfected = df[df.index % 4 == 1]
dfRecovered = df[df.index % 4 == 2]
dfDead = df[df.index % 4 == 3]
dfInfected.head()

### Controll

We check three of the simulations to make sure the data seems correct.

In [ ]:
# Plot three of the simulations
fig, ax = plt.subplots()

ax.plot(range(dfInfected.shape[1]), dfInfected.loc[1,:], label="simulation_1")
ax.plot(range(dfInfected.shape[1]), dfInfected.loc[5,:], label="simulation_2")
ax.plot(range(dfInfected.shape[1]), dfInfected.loc[9,:], label="simulation_3")

ax.set_title("Three first simulations")
ax.set_xlabel("iterations")
ax.set_ylabel("")
ax.legend()

plt.show()

### Normalizing

We normalize and split the data to prepare it for training and testing.

In [ ]:
dfInfected_arr = dfInfected.values
dfInfected_arr.shape
TRAIN_SPLIT = int(dfInfected_arr.shape[0]-dfInfected_arr.shape[0]*0.1)
print(TRAIN_SPLIT)

uni_train_mean = dfInfected_arr[:TRAIN_SPLIT].mean()
uni_train_std = dfInfected_arr[:TRAIN_SPLIT].std()
uni_data = (dfInfected_arr-uni_train_mean)/uni_train_std
print ('\nUnivariate data shape')
print(uni_data.shape)

## Creating a TimeSeries

We now need to create a TimeSeries to use later on.

In [ ]:
def time_series(dataset: np.ndarray, start_series: int, end_series: int, history_size: int = 20, target_size: int = 1) -> tuple[np.ndarray, np.ndarray]:
    '''
    Creates TimeSeries of sequential data to use with RNN training and predicting.
    
    Args:
        dataset: the data to create TimeSeries from
        start_series: index for the first entry
        end_series: index for the last entry
        history_size: number of past data to use in prediction
        target_size: amount of predictions to perform
    
    Returns:
        time_series, actual_values
    '''

    series = []
    actual_values = []

    start_index = history_size
    end_index = dataset.shape[1] - target_size

    for c in range(start_series, end_series):
        for i in range(start_index, end_index):
            indices = np.arange(i - history_size, i)
            series.append(dataset[c][indices].reshape((history_size, 1)))
            actual_values.append(dataset[c][i + target_size])
    
    return np.array(series), np.array(actual_values)


In [ ]:
univariate_past_history = 20 # Days
univariate_future_target = 0 # Current day

x_train_uni, y_train_uni = time_series(uni_data, 0, TRAIN_SPLIT,
                                           univariate_past_history,
                                           univariate_future_target)
x_val_uni, y_val_uni = time_series(uni_data, TRAIN_SPLIT, len(uni_data),
                                       univariate_past_history,
                                       univariate_future_target)

In [ ]:
print ('Single window of past history')
print (x_train_uni[0])
print ('\n Target number to predict')
print (y_train_uni[0])
print ('\n Number of traing data points')
print (y_train_uni.shape[0])
print ('\n Number of test data points')
print (x_val_uni.shape[0])

In [ ]:
def create_time_steps(length):
    return [i for i in range(-length, 0)]

def show_plot(plot_data: np.ndarray, delta: float, title: str):
    labels = ['History', 'True Future', 'Model Prediction']
    marker = ['.-', 'rx', 'go']
    time_steps = create_time_steps(plot_data[0].shape[0])

    fig, ax = plt.subplots()
    if delta:
        future = delta
    else:
        future = 0
    ax.set_title(title)
    for i, x in enumerate(plot_data):
        if i:
            ax.plot(future, plot_data[i], marker[i], markersize=10,label=labels[i])
        else:
            ax.plot(time_steps, plot_data[i].flatten(), marker[i], label=labels[i])
    ax.legend()
    ax.set_xlim([time_steps[0], (future+5)*2])
    ax.set_xlabel('Time-Step')
    return fig, ax

def show_dual_plot(plot_data1: np.ndarray, plot_data2: np.ndarray, delta: float, title1: str, title2: str):
    labels = ['History', 'True Future', 'Model Prediction']
    marker = ['.-', 'rx', 'go']
    time_steps1 = create_time_steps(plot_data1[0].shape[0])
    time_steps2 = create_time_steps(plot_data2[0].shape[0])

    fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2)
    if delta:
        future = delta
    else:
        future = 0
    ax1.set_title(title1)
    ax2.set_title(title2)
    for i, x in enumerate(plot_data1):
        if i:
            ax1.plot(future, plot_data1[i], marker[i], markersize=10,label=labels[i])
        else:
            ax1.plot(time_steps1, plot_data1[i].flatten(), marker[i], label=labels[i])
    for i, x in enumerate(plot_data2):
        if i:
            ax2.plot(future, plot_data2[i], marker[i], markersize=10,label=labels[i])
        else:
            ax2.plot(time_steps2, plot_data2[i].flatten(), marker[i], label=labels[i])
    ax1.legend()
    ax2.legend()
    ax1.set_xlim([time_steps1[0], (future+5)*2])
    ax2.set_xlim([time_steps2[0], (future+5)*2])
    ax1.set_xlabel('Time-Step')
    ax2.set_xlabel('Time-Step')
    return fig, (ax1, ax2)

In [ ]:
fig, ax = show_plot([x_train_uni[0], y_train_uni[0]], 0, 'Sample Example')
plt.show()

### Baseline Forecasting

To get a baseline we simply predict the mean of the history.

In [ ]:

def baseline(history):
    return np.mean(history)

In [ ]:
fig, ax = show_plot([x_train_uni[0], y_train_uni[0], baseline(x_train_uni[0])], 0, 'Baseline Prediction Example')
plt.show()

Not a very good guess in this case.

## Normalized LSTM based Forecasting

Now we implement a LSTM model and use it to predict the next value.

In [ ]:
print (x_train_uni.shape)
print (y_train_uni.shape)
x_train_uni.dtype

In [ ]:
BATCH_SIZE = 256
BUFFER_SIZE = 10000

train_univariate = tf.data.Dataset.from_tensor_slices((x_train_uni, y_train_uni))
train_univariate = train_univariate.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE).repeat()

val_univariate = tf.data.Dataset.from_tensor_slices((x_val_uni, y_val_uni))
val_univariate = val_univariate.batch(BATCH_SIZE).repeat()

print(train_univariate)

## Initiating the simple LSTM model

2 layers with one having 8 neurons and one dense layer.

In [ ]:
simple_lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(8, input_shape=x_train_uni.shape[-2:]),
    tf.keras.layers.Dense(1)
])

simple_lstm_model.compile(optimizer='adam', loss='mae')
simple_lstm_model.summary()
print(x_train_uni.shape[-2:])

In [ ]:
for x, y in val_univariate.take(1):
    print(x.shape)
    print(simple_lstm_model.predict(x).shape)
    print(y.shape)

### Training

In [ ]:
EVALUATION_INTERVAL = 2000
EPOCHS = 10

simple_lstm_model.fit(train_univariate, 
                      epochs=EPOCHS,
                      steps_per_epoch=EVALUATION_INTERVAL,
                      validation_data=val_univariate, 
                      validation_steps=50)

### Predicting

In [ ]:
for x, y in val_univariate.take(3):
    fig, ax = show_plot([x[0].numpy(), y[0].numpy(), simple_lstm_model.predict(x)[0]], 0, 'Simple LSTM model')
    plt.show()

## Initiating one deeper and one wider LSTM model

4 layers with 3 having 16 -> 8 -> 4 neurons and one dense output.

The deeper model might be able to catch more complex patterns from the data while the wider one might be better suited since we don't have a lot of data.

In [ ]:
deeper_lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(16, return_sequences=True, input_shape=x_train_uni.shape[-2:]),
    tf.keras.layers.LSTM(8, return_sequences=True),
    tf.keras.layers.LSTM(4),
    tf.keras.layers.Dense(1)
])

wider_lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(32, input_shape=x_train_uni.shape[-2:]),
    tf.keras.layers.Dense(1)
])

deeper_lstm_model.compile(optimizer='adam', loss='mae')
deeper_lstm_model.summary()

wider_lstm_model.compile(optimizer='adam', loss='mae')
wider_lstm_model.summary()

In [ ]:
for x, y in val_univariate.take(1):
    print(x.shape)
    print(deeper_lstm_model.predict(x).shape)
    print(wider_lstm_model.predict(x).shape)
    print(y.shape)

### Training

In [ ]:
EVALUATION_INTERVAL_DEEPER = 1000
EVALUATION_INTERVAL_WIDER = 2000
EPOCHS = 10

deeper_lstm_model.fit(train_univariate, 
                      epochs=EPOCHS,
                      steps_per_epoch=EVALUATION_INTERVAL_DEEPER,
                      validation_data=val_univariate, 
                      validation_steps=50)

wider_lstm_model.fit(train_univariate, 
                      epochs=EPOCHS,
                      steps_per_epoch=EVALUATION_INTERVAL_WIDER,
                      validation_data=val_univariate, 
                      validation_steps=50)

### Predicting

In [ ]:
for x, y in val_univariate.take(3):
    fig, (ax1, ax2) = show_dual_plot([x[0].numpy(),
                                      y[0].numpy(),
                                      deeper_lstm_model.predict(x)[0]],
                                      [x[0].numpy(),
                                       y[0].numpy(),
                                       wider_lstm_model.predict(x)[0]],
                                      0,
                                      'Deeper LSTM model',
                                      'Wider LSTM model')
    plt.show()

### Findings

It turns out that the deeper model might have either overfit on the data or that the gradients vanished along the training since validation loss stagnated around epoch 4.

The wider model on the other hand was seemingly worse until epoch 10 when it passed the simple model in validation loss and hit the predictions almost spot on.

The parameter use was a lot higher in these two models, both more than 4x as many parameters than the simple model with longer training times to boot.

## Multivariate LSTM based forecasting - Single Step

This time we will be using three inputs instead to predict the number of infected people. (infected, recovered, deceased)

In [ ]:
dfInfected.loc[1,:].plot(label='Infected')
dfRecovered.loc[2,:].plot(label='Recovered')
dfDead.loc[3,:].plot(label='Dead')
dfInfected_arr = dfInfected.values
dfRecovered_arr = dfRecovered.values
dfDead_arr = dfDead.values
plt.legend()
plt.show()

### Preparing data

In [ ]:
dfInfected_train_mean = dfInfected_arr[:TRAIN_SPLIT].mean()
dfInfected_train_std = dfInfected_arr[:TRAIN_SPLIT].std()
dfInfected_data = (dfInfected_arr-dfInfected_train_mean)/dfInfected_train_std
#for Recovered
dfRecovered_train_mean = dfRecovered_arr[:TRAIN_SPLIT].mean()
dfRecovered_train_std = dfRecovered_arr[:TRAIN_SPLIT].std()
dfRecovered_data = (dfRecovered_arr-dfRecovered_train_mean)/dfRecovered_train_std
#for Dead
dfDead_train_mean = dfDead_arr[:TRAIN_SPLIT].mean()
dfDead_train_std = dfDead_arr[:TRAIN_SPLIT].std()
dfDead_data = (dfDead_arr-dfDead_train_mean)/dfDead_train_std

In [ ]:
dataset = np.array([dfInfected_data, dfRecovered_data, dfDead_data])
dataset.shape
print ('\n Multivariate data shape')
print(dataset.shape)

### Creating multivariate TimeSeries

In [ ]:
def multivariate_time_series(dataset: np.ndarray, target: np.ndarray, start_series: int, end_series: int, history_size: int = 20, target_size: int = 1, step: int = 2, single_step: bool = False) -> tuple[np.ndarray, np.ndarray]:
    '''
    Creates multivariate TimeSeries of sequential data to use with RNN training and predicting.
    
    Args:
        dataset: data to create TimeSeries from
        target: data to predict
        start_series: index for the first entry
        end_series: index for the last entry
        history_size: number of past data to use in prediction
        target_size: amount of predictions to perform
        step: how large of a step to take
        single_step: take one step or not
    
    Returns:
        time_series, actual_values
    '''

    data = []
    labels = []
    start_index = history_size
    end_index = len(dataset[0][0]) - target_size
    for c in range(start_series, end_series):
        for i in range(start_index, end_index):
            indices = range(i-history_size, i, step)
            one = dataset[0][c][indices]
            two = dataset[1][c][indices]
            three = dataset[2][c][indices]
            data.append(np.transpose(np.array([one, two, three])))
            
            if single_step:
                labels.append(target[c][i+target_size])
            else:
                labels.append(np.transpose(target[c][i:i+target_size]))
    return np.array(data), np.array(labels)


In [ ]:
past_history = 20
future_target = 5
STEP = 2

x_train_single, y_train_single = multivariate_time_series(dataset, dfInfected_data, 0, TRAIN_SPLIT, 
                                                   past_history, future_target, STEP,
                                                   single_step=True)
x_val_single, y_val_single = multivariate_time_series(dataset, dfInfected_data, TRAIN_SPLIT, dataset.shape[1], 
                                               past_history, future_target, STEP,
                                               single_step=True)

In [ ]:
print ('Single window of past history : {}'.format(x_train_single[0].shape))
print(dataset.shape)

In [ ]:
train_data_single = tf.data.Dataset.from_tensor_slices((x_train_single, y_train_single))
train_data_single = train_data_single.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE).repeat()

val_data_single = tf.data.Dataset.from_tensor_slices((x_val_single, y_val_single))
val_data_single = val_data_single.batch(BATCH_SIZE).repeat()

### Plotting tools

In [ ]:
def plot_train_history(history, title):
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(len(loss))
    plt.figure()
    plt.plot(epochs, loss, 'b', label='Training loss')
    plt.plot(epochs, val_loss, 'r', label='Validation loss')
    plt.title(title)
    plt.legend()
    plt.show()

def plot_dual_train_history(history1, history2, title1, title2):
    loss1 = history1.history['loss']
    loss2 = history2.history['loss']
    val_loss1 = history1.history['val_loss']
    val_loss2 = history2.history['val_loss']
    epochs1 = range(len(loss1))
    epochs2 = range(len(loss2))
    plt.figure()
    plt.subplot(1, 2, 1)
    plt.plot(epochs1, loss1, 'b', label='Training loss')
    plt.plot(epochs1, val_loss1, 'r', label='Validation loss')
    plt.title(title1)
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs2, loss2, 'b', label='Training loss')
    plt.plot(epochs2, val_loss2, 'r', label='Validation loss')
    plt.title(title1)
    plt.legend()
    plt.show()

## Initiating the simple model

This time we use the wide layout for the simple model.

In [ ]:
single_step_model = tf.keras.models.Sequential()
single_step_model.add(tf.keras.layers.LSTM(32, input_shape=x_train_single.shape[-2:]))
single_step_model.add(tf.keras.layers.Dense(1))

single_step_model.compile(optimizer=tf.keras.optimizers.RMSprop(), loss='mae')
single_step_model.summary()
x_train_single.shape[-2:]

In [ ]:
for x, y in val_data_single.take(1):
    print(single_step_model.predict(x).shape)
print ('\n Number of traing data points')
print (x_train_single.shape[0])
print ('\n Number of test data points')
print (x_val_single.shape[0])

### Training

In [ ]:
single_step_history = single_step_model.fit(train_data_single, epochs=EPOCHS,
                                            steps_per_epoch=EVALUATION_INTERVAL,
                                            validation_data=val_data_single,
                                            validation_steps=50)

In [ ]:
plot_train_history(single_step_history,'Single Step Training and validation loss')

### predicting

In [ ]:
for x, y in val_data_single.take(3):
    fig, ax = show_plot([x[0][:, 0].numpy(), y[0].numpy(),
                    single_step_model.predict(x)[0]], future_target,
                   'Single Step Prediction')
    plt.show()

## Initiating one deeper and one wider multivariate LSTM model

Here we let the deeper model have LSTM layers 32 -> 16 -> 8 and a dense output, while the wider one will have one 128 neuron wide LSTM layer with a dense output layer.

The idea is the same here as before, hoping the deeper model can find those deeper patters in the simulations while the wider should find the complex patters and be quicker to train.

In [ ]:
deeper_single_step_lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(64, return_sequences=True, input_shape=x_train_single.shape[-2:]),
    tf.keras.layers.LSTM(32, return_sequences=True),
    tf.keras.layers.LSTM(16),
    tf.keras.layers.Dense(1)
])

wider_single_step_lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(128, input_shape=x_train_single.shape[-2:]),
    tf.keras.layers.Dense(1)
])

deeper_single_step_lstm_model.compile(optimizer=tf.keras.optimizers.RMSprop(), loss='mae')
deeper_single_step_lstm_model.summary()

wider_single_step_lstm_model.compile(optimizer=tf.keras.optimizers.RMSprop(), loss='mae')
wider_single_step_lstm_model.summary()

### Training

In [ ]:
deeper_single_step_history = deeper_single_step_lstm_model.fit(train_data_single, epochs=EPOCHS,
                                            steps_per_epoch=EVALUATION_INTERVAL_DEEPER,
                                            validation_data=val_data_single,
                                            validation_steps=50)

wider_single_step_history = wider_single_step_lstm_model.fit(train_data_single, epochs=EPOCHS,
                                            steps_per_epoch=EVALUATION_INTERVAL_WIDER,
                                            validation_data=val_data_single,
                                            validation_steps=50)

In [ ]:
plot_dual_train_history(deeper_single_step_history, wider_single_step_history, 'Deeper Single Step', 'Wider Single Step')

### Predictions

In [ ]:
for x, y in val_data_single.take(3):
    fig, (ax1, ax2) = show_dual_plot([x[0][:, 0].numpy(),
                                      y[0].numpy(),
                                      deeper_single_step_lstm_model.predict(x)[0]],
                                      [x[0][:, 0].numpy(),
                                      y[0].numpy(),
                                      wider_single_step_lstm_model.predict(x)[0]],
                                      future_target,
                                      'Deeper Single Step Prediction',
                                      'Wider Single Step Prediction')
    plt.show()

### Findings

In this case the difference is not as clear cut as in the univariate case.
Here the two different models seem to perform roughly as good as the simple model, although the simple model beats both in validation_loss after 10 epochs, with the wider model not far behind.

The number om parameters used in this case skyrocketed from the simple and modest ca 4 000 parameters, to the deeper and medium ca 33 000 parameters, and with the wide hulking ca 68 000 parameters.

This of course affected training times negatively with the wider model taking roughly 4.5 times longer to train, and the deeper model had to have it's steps cut in half since it had large overfit issues with 2 000 steps, but would have taken 4 times longer than the simple model.

## Multivariate LSTM - Multiple Steps

This time we are still using three inputs to predict the simulation, but now we will predict more than one day.

This will be done in two different ways:

+ One way will be to predict all the days at once
+ While the other will be to predict one value after the other and use the predicted values to predict the next values iteratively.

### Preparing the data

In [ ]:
# data for multi-predictor
past_history = 40
future_target_multi = 10
STEP =2
x_train_multi, y_train_multi = multivariate_time_series(dataset, dfInfected_data, 0, TRAIN_SPLIT, 
                                                    past_history, future_target_multi, STEP)
x_val_multi, y_val_multi = multivariate_time_series(dataset, dfInfected_data, TRAIN_SPLIT, dataset.shape[1], 
                                                past_history, future_target_multi, STEP)

# data for iterative-predictor
past_history = 40
STEP =2
x_train_infected_iterative, y_train_infected_iterative = multivariate_time_series(dataset, dfInfected_data, 0, TRAIN_SPLIT, 
                                                    past_history, 1, STEP)
x_val_infected_iterative, y_val_infected_iterative = multivariate_time_series(dataset, dfInfected_data, TRAIN_SPLIT, dataset.shape[1], 
                                                past_history, 1, STEP)
x_train_recovered_iterative, y_train_recovered_iterative = multivariate_time_series(dataset, dfRecovered_data, 0, TRAIN_SPLIT, 
                                                    past_history, 1, STEP)
x_val_recovered_iterative, y_val_recovered_iterative = multivariate_time_series(dataset, dfRecovered_data, TRAIN_SPLIT, dataset.shape[1], 
                                                past_history, 1, STEP)
x_train_deceased_iterative, y_train_deceased_iterative = multivariate_time_series(dataset, dfDead_data, 0, TRAIN_SPLIT, 
                                                    past_history, 1, STEP)
x_val_deceased_iterative, y_val_deceased_iterative = multivariate_time_series(dataset, dfDead_data, TRAIN_SPLIT, dataset.shape[1], 
                                                past_history, 1, STEP)

In [ ]:
train_data_multi = tf.data.Dataset.from_tensor_slices((x_train_multi, y_train_multi))
train_data_multi = train_data_multi.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE).repeat()

val_data_multi = tf.data.Dataset.from_tensor_slices((x_val_multi, y_val_multi))
val_data_multi = val_data_multi.batch(BATCH_SIZE).repeat()

# Infected
train_data_infected_iterative = tf.data.Dataset.from_tensor_slices((x_train_infected_iterative, y_train_infected_iterative))
train_data_infected_iterative = train_data_infected_iterative.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE).repeat()

val_data_infected_iterative = tf.data.Dataset.from_tensor_slices((x_val_infected_iterative, y_val_infected_iterative))
val_data_infected_iterative = val_data_infected_iterative.batch(BATCH_SIZE).repeat()

# Recovered
train_data_recovered_iterative = tf.data.Dataset.from_tensor_slices((x_train_recovered_iterative, y_train_recovered_iterative))
train_data_recovered_iterative = train_data_recovered_iterative.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE).repeat()

val_data_recovered_iterative = tf.data.Dataset.from_tensor_slices((x_val_recovered_iterative, y_val_recovered_iterative))
val_data_recovered_iterative = val_data_recovered_iterative.batch(BATCH_SIZE).repeat()

# Deceased
train_data_deceased_iterative = tf.data.Dataset.from_tensor_slices((x_train_deceased_iterative, y_train_deceased_iterative))
train_data_deceased_iterative = train_data_deceased_iterative.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE).repeat()

val_data_deceased_iterative = tf.data.Dataset.from_tensor_slices((x_val_deceased_iterative, y_val_deceased_iterative))
val_data_deceased_iterative = val_data_deceased_iterative.batch(BATCH_SIZE).repeat()

### Plotting tools

In [ ]:
def multi_step_plot(history, true_future, prediction):
    plt.figure(figsize=(12, 6))
    num_in = create_time_steps(len(history))
    num_out = len(true_future)
    plt.plot(num_in, np.array(history[:, 0]), label='History')
    plt.plot(np.arange(num_out), np.array(true_future), 'bo', label='True Future')
    if prediction.any():
        plt.plot(np.arange(num_out), np.array(prediction), 'ro', label='Predicted Future')
    plt.legend(loc='upper left')
    plt.show()

def iterative_step_plot(history, true_future, prediction):
    names = ['Infected', 'Recovered', 'Deceased']
    colors = ['r', 'g', 'k']
    plt.figure(figsize=(12, 6))
    num_in = create_time_steps(history.shape[1])
    for col in range(prediction.shape[1]):
        num_out = true_future.shape[1]
        plt.plot(num_in, np.array(history[0, :, col]), f'{colors[col]}-', label=f'{names[col]} History')
        # plt.plot(np.arange(num_out), true_future.flatten(), f'{colors[col]}o', label=f'True {names[col]} Future')
        if prediction.T[col].any():
            plt.plot(np.arange(prediction.shape[0]), prediction[:, col], f'{colors[col]}o', label='Predicted Future')
    plt.legend(loc='upper left')
    plt.show()

In [ ]:
for x, y in train_data_multi.take(1):
    multi_step_plot(x[0], y[0], np.array([0]))

## Initialising the multi-predictor LSTM model

The multi-predictor model will have 2 LSTM layers 32 -> 16 and a dense output layer.

In [ ]:
multi_step_model = tf.keras.models.Sequential()
multi_step_model.add(tf.keras.layers.LSTM(32,
                                          return_sequences=True,
                                          input_shape=x_train_multi.shape[-2:]))
multi_step_model.add(tf.keras.layers.LSTM(16, activation='relu'))
multi_step_model.add(tf.keras.layers.Dense(future_target_multi))

multi_step_model.compile(optimizer=tf.keras.optimizers.RMSprop(clipvalue=1.0), loss='mae')
multi_step_model.summary()
x_train_multi.shape[-2:]

In [ ]:
for x, y in val_data_multi.take(1):
    print (multi_step_model.predict(x).shape)

### Training

In [ ]:
multi_step_history = multi_step_model.fit(train_data_multi, epochs=EPOCHS,
                                          steps_per_epoch=EVALUATION_INTERVAL,
                                          validation_data=val_data_multi,
                                          validation_steps=50)

In [ ]:
plot_train_history(multi_step_history, 'Multi-Step Training and validation loss')

### Predicting

In [ ]:

for x, y in val_data_multi.take(3):
    multi_step_plot(x[0], y[0], multi_step_model.predict(x)[0])

## Initializing the iterative-predictor LSTM model

This model will have the same layers as the multi-predictor (32 (LSTM) -> 16 (LSTM) -> 1 (d)) but will only train on predicting one value instead of the next 10, but it will predict the next value 10 times, using the previous prediction as the next input data.

In [ ]:
iterative_step_model_infected = tf.keras.models.Sequential()
iterative_step_model_infected.add(tf.keras.layers.LSTM(32,
                                  return_sequences=True,
                                  input_shape=x_train_infected_iterative.shape[-2:]))
iterative_step_model_infected.add(tf.keras.layers.LSTM(16, activation='relu'))
iterative_step_model_infected.add(tf.keras.layers.Dense(1))

iterative_step_model_infected.compile(optimizer=tf.keras.optimizers.RMSprop(clipvalue=1.0), loss='mae')
iterative_step_model_infected.summary()


iterative_step_model_recovered = tf.keras.models.Sequential()
iterative_step_model_recovered.add(tf.keras.layers.LSTM(32,
                                   return_sequences=True,
                                   input_shape=x_train_recovered_iterative.shape[-2:]))
iterative_step_model_recovered.add(tf.keras.layers.LSTM(16, activation='relu'))
iterative_step_model_recovered.add(tf.keras.layers.Dense(1))

iterative_step_model_recovered.compile(optimizer=tf.keras.optimizers.RMSprop(clipvalue=1.0), loss='mae')
iterative_step_model_recovered.summary()


iterative_step_model_deceased = tf.keras.models.Sequential()
iterative_step_model_deceased.add(tf.keras.layers.LSTM(32,
                                  return_sequences=True,
                                  input_shape=x_train_deceased_iterative.shape[-2:]))
iterative_step_model_deceased.add(tf.keras.layers.LSTM(16, activation='relu'))
iterative_step_model_deceased.add(tf.keras.layers.Dense(1))

iterative_step_model_deceased.compile(optimizer=tf.keras.optimizers.RMSprop(clipvalue=1.0), loss='mae')
iterative_step_model_deceased.summary()

In [ ]:
for x, y in val_data_infected_iterative.take(1):
    print (iterative_step_model_infected.predict(x).shape)
    print (x.shape)

### Training

In [ ]:
iterative_step_history_infected = iterative_step_model_infected.fit(
    train_data_infected_iterative, 
    epochs=EPOCHS,
    steps_per_epoch=EVALUATION_INTERVAL,
    validation_data=val_data_infected_iterative,
    validation_steps=50
)

iterative_step_history_recovered = iterative_step_model_recovered.fit(
    train_data_recovered_iterative, 
    epochs=EPOCHS,
    steps_per_epoch=EVALUATION_INTERVAL,
    validation_data=val_data_recovered_iterative,
    validation_steps=50
)

iterative_step_history_deceased = iterative_step_model_deceased.fit(
    train_data_deceased_iterative, 
    epochs=EPOCHS,
    steps_per_epoch=EVALUATION_INTERVAL,
    validation_data=val_data_deceased_iterative,
    validation_steps=50
)

### Predicting

Here we need to create a special prediction function to implement iterative prediction.

In [ ]:
def iterative_predict(model_infected, model_recovered, model_deceased, initial_sequence, steps_to_predict):
    """
    Predict multiple steps one at a time
    """
    predictions = []
    current_sequence = tf.reshape(tf.identity(initial_sequence), (1, initial_sequence.shape[0], initial_sequence.shape[1]))  # Create a copy
    
    for _ in range(steps_to_predict):
        
        # Predict next value (runs eagerly)
        next_infected_val = model_infected.predict(current_sequence, verbose=False)[0]
        next_recovered_val = model_recovered.predict(current_sequence, verbose=False)[0]
        next_deceased_val = model_deceased.predict(current_sequence, verbose=False)[0]
        predictions.append([next_infected_val[0], next_recovered_val[0], next_deceased_val[0]])

        next_tensor = tf.reshape(tf.convert_to_tensor([next_infected_val, next_recovered_val, next_deceased_val], tf.double), (1, 1, 3))

        # Remove first element, append new value
        current_sequence = tf.concat([
            current_sequence[:, 1:, :],  # All except first [samples, timesteps, features]
            next_tensor
        ], axis=1)
    
    return np.array(predictions)  # [[infected, recovered, deceased], ...]

In [ ]:
iterations = 5
for (x1, y1), (x2, y2), (x3, y3) in zip(val_data_infected_iterative.take(3), val_data_recovered_iterative.take(3), val_data_deceased_iterative.take(3)):
    x1 = tf.reshape(x1[0], (1, 20, 3))
    iterative_step_plot(x, tf.concat([y1[0], y2[0], y3[0]], axis=0).numpy().reshape(1, -1), iterative_predict(iterative_step_model_infected,
                            iterative_step_model_recovered,
                            iterative_step_model_deceased,
                            x[0], iterations))